# Project Aura 2.0 — Notebook 07: Deployment Preparation

## Overview

Prepares Project Aura for deployment: configuration, FastAPI backend, Pydantic schemas, health/readiness, WebSocket prep, Docker. Does NOT claim deployment has occurred.

## Sections
1-15. See cells below.

In [1]:
!pip install --quiet fastapi uvicorn pydantic httpx psutil mediapipe opencv-python-headless numpy pillow matplotlib
print("Dependencies installed for NB07")

Dependencies installed for NB07



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Deployment Overview

## 2. Architecture Inspection

In [2]:
import os, sys
# Add parent directory to path for src imports
parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
for p in ["/content/project-aura"]:
    if p not in sys.path: sys.path.insert(0, p)
for root, dirs, files in os.walk("src"):
    for f in files:
        if f.endswith(".py"): print(f"  {os.path.join(root, f)}")

  src\__init__.py
  src\agents\action_registry.py
  src\agents\context.py
  src\agents\decision_engine.py
  src\agents\executor.py
  src\agents\gesture.py
  src\agents\observation.py
  src\agents\orchestrator.py
  src\agents\permission_manager.py
  src\agents\verification.py
  src\agents\__init__.py
  src\api\config.py
  src\api\main.py
  src\api\routes.py
  src\api\__init__.py
  src\api\routes\agent.py
  src\api\routes\gesture.py
  src\api\routes\health.py
  src\api\routes\inference.py
  src\api\routes\__init__.py
  src\api\schemas\agent.py
  src\api\schemas\gesture.py
  src\api\schemas\health.py
  src\api\schemas\inference.py
  src\api\schemas\__init__.py
  src\api\services\agent_service.py
  src\api\services\gesture_service.py
  src\api\services\model_service.py
  src\api\services\__init__.py
  src\core\config.py
  src\core\constants.py
  src\core\exceptions.py
  src\core\__init__.py
  src\dashboard\app.py
  src\monitoring\analytics.py
  src\monitoring\logger.py
  src\monitoring\met

## 3. Configuration

In [3]:
from src.core.config import load_config
from src.api.config import load_api_config
config = load_config(); api_config = load_api_config()
print(f"Model: {config.vision.model_name}, GPU: {config.colab.gpu_enabled}")
print(f"API: {api_config.host}:{api_config.port}, env={api_config.environment}")

Model: efficientnet_b0, GPU: False
API: 0.0.0.0:8000, env=development


## 4. API Schemas

In [4]:
from src.api.schemas import HealthResponse, ReadinessResponse, ModelInfoResponse, InferenceRequest, InferenceResponse, GestureRequest, GestureResponse, AgentDecideRequest, AgentDecideResponse
for cls in [HealthResponse, ReadinessResponse, ModelInfoResponse, InferenceRequest, InferenceResponse, GestureRequest, GestureResponse, AgentDecideRequest, AgentDecideResponse]:
    print(f"  {cls.__name__}: {len(cls.model_fields)} fields")

  HealthResponse: 4 fields


  ReadinessResponse: 3 fields
  ModelInfoResponse: 5 fields
  InferenceRequest: 3 fields
  InferenceResponse: 4 fields
  GestureRequest: 2 fields
  GestureResponse: 4 fields
  AgentDecideRequest: 5 fields
  AgentDecideResponse: 11 fields


## 5. Model Service

In [5]:
from src.api.services.model_service import ModelService
ms = ModelService(model_name="efficientnet_b0")
print(f"Model: {ms.model_name}, loaded={ms.is_loaded()}")
print(f"Info: {ms.get_model_info()}")

Model: efficientnet_b0, loaded=False
Info: {'model_name': 'efficientnet_b0', 'num_classes': 10, 'loaded': False, 'device': 'cpu'}


## 6. Inference Route

In [6]:
from src.api.routes.inference import router as ir
for r in ir.routes: print(f"  {r.methods} {r.path}")

  {'POST'} /inference


## 7. Gesture Route

In [7]:
from src.api.routes.gesture import router as gr
for r in gr.routes: print(f"  {r.methods} {r.path}")
from src.api.services.gesture_service import GestureService
print(f"Available: {GestureService().is_available()}")

  {'POST'} /gesture


Available: False


## 8. Agent Route

In [8]:
from src.api.routes.agent import router as ar
for r in ar.routes:
    print(f"  {getattr(r,'methods',['WS'])} {r.path}")
from src.api.services.agent_service import AgentService
result = AgentService().decide({"observation_source":"test","observation_type":"gesture","data":{"gesture":"OPEN_PALM"},"confidence":0.9})
print(f"Agent test: {result['selected_action']}")

  {'POST'} /agent/decide
  ['WS'] /ws/gesture
Agent test: pause


## 9. Health & Readiness

In [9]:
from src.api.routes.health import router as hr
for r in hr.routes: print(f"  {r.methods} {r.path}")

  {'GET'} /health
  {'GET'} /ready
  {'GET'} /model/info


## 10. WebSocket Prep

`/ws/gesture` scaffold. Not production WebRTC.

In [10]:
print("WebSocket: /ws/gesture (scaffold)")
print("Production WebRTC NOT implemented")

WebSocket: /ws/gesture (scaffold)
Production WebRTC NOT implemented


## 11. API Testing

In [11]:
from fastapi.testclient import TestClient
from src.api.main import app
client = TestClient(app)
print(f"GET /health: {client.get('/health').status_code}")
print(f"GET /ready: {client.get('/ready').status_code}")
print(f"GET /model/info: {client.get('/model/info').status_code}")
r = client.post("/agent/decide", json={"observation_source":"test","observation_type":"gesture","data":{"gesture":"OPEN_PALM"},"confidence":0.9})
print(f"POST /agent/decide: {r.status_code} - {r.json()['selected_action']}")
r = client.post("/inference", json={"image_base64":"bad","format":"jpeg"})
print(f"POST /inference (bad): {r.status_code}")
print("All API tests passed")

GET /health: 200


GET /ready: 200

GET /model/info: 200
POST /agent/decide: 200 - pause
POST /inference (bad): 400
All API tests passed


## 12. Docker Preparation

In [12]:
dockerfile = "FROM python:3.11-slim\nWORKDIR /app\nRUN apt-get update && apt-get install -y libgl1-mesa-glx libglib2.0-0 libsm6 libxrender1 libxext6 && rm -rf /var/lib/apt/lists/*\nCOPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\nCOPY src/ ./src/\nEXPOSE 8000\nCMD [\"uvicorn\", \"src.api.main:app\", \"--host\", \"0.0.0.0\", \"--port\", \"8000\"]\n"
reqs = "numpy>=1.24.0\nopencv-python-headless>=4.8.0\npillow>=10.0.0\nfastapi>=0.100.0\nuvicorn>=0.23.0\npydantic>=2.0.0\nmediapipe>=0.10.0\npsutil>=5.9.0\nmatplotlib>=3.7.0\n"
os.makedirs("deploy", exist_ok=True)
open("deploy/Dockerfile","w").write(dockerfile)
open("deploy/requirements.txt","w").write(reqs)
print("Dockerfile + requirements in deploy/ (CPU-compatible)")

Dockerfile + requirements in deploy/ (CPU-compatible)


## 13. Deployment Instructions

In [13]:
print("1. pip install -r deploy/requirements.txt")
print("2. cp .env.example .env")
print("3. uvicorn src.api.main:app --host 0.0.0.0 --port 8000")
print("4. pytest tests/test_api.py -v")
print("5. docker build -t project-aura:2.0 -f deploy/Dockerfile .")
print("6. docker run -p 8000:8000 project-aura:2.0")
print("NOTE: NOT deployed to any external service.")

1. pip install -r deploy/requirements.txt
2. cp .env.example .env
3. uvicorn src.api.main:app --host 0.0.0.0 --port 8000
4. pytest tests/test_api.py -v
5. docker build -t project-aura:2.0 -f deploy/Dockerfile .
6. docker run -p 8000:8000 project-aura:2.0
NOTE: NOT deployed to any external service.


## 14. Validation

In [14]:
for path, desc in [("deploy/Dockerfile","Dockerfile"),("deploy/requirements.txt","requirements"),("src/api/main.py","FastAPI"),("src/api/config.py","config"),("src/api/schemas/","schemas"),("src/api/routes/","routes"),("src/api/services/","services")]:
    print(f"  [{'OK' if os.path.exists(path) else 'MISSING'}] {desc}")
from src.api.main import app
for r in app.routes:
    if hasattr(r, "methods"): print(f"  {r.methods} {r.path}")

  [OK] Dockerfile
  [OK] requirements
  [OK] FastAPI
  [OK] config
  [OK] schemas
  [OK] routes
  [OK] services
  {'HEAD', 'GET'} /openapi.json
  {'HEAD', 'GET'} /docs
  {'HEAD', 'GET'} /docs/oauth2-redirect
  {'HEAD', 'GET'} /redoc


## 15. Final Summary

FastAPI backend, schemas, lazy loading, WebSocket scaffold, Docker prep. NOT deployed.